### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="magicbricks_house_price",
    dataset_year="2023",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/juhibhojani/house-price",
    download_description=r"""We download the data from Kaggle.

kaggle datasets download juhibhojani/house-price && unzip house-price.zip && rm house-price.zip
mkdir -p local-data-warehouse/magicbricks_house_price && mv house_prices.csv local-data-warehouse/magicbricks_house_price/
""",
    # References
    academic_reference_bibtex=r"""@misc{Bhojani2023Magicbricks,
  author = {Juhi Bhojani},
  title = {Magicbricks House Price},
  year = {2023},
  howpublished = {\url{https://www.kaggle.com/datasets/juhibhojani/house-price}},
  note = {Kaggle dataset}
}
""",
    academic_reference_bibtex_key="Bhojani2023Magicbricks",
    license="Community Data License Agreement - Sharing - Version 1.0",
    data_tags=["IID", "ForcedIIDFromTemporal"],
    curation_comments="""Looks reasonable but yet another house price dataset.
Missing time information — treated as IID although data is likely non-IID by nature.

Preprocessing notes:
- 'Amount(in rupees)' is stored as strings with Indian unit suffixes (e.g. '42 Lac', '1.5 Cr')
  and is parsed to numeric rupees (1 Lac = 100_000; 1 Cr = 10_000_000).
- 'Carpet Area' and 'Super Area' are stored as strings with 'sqft' suffix and parsed to float.
- 'Floor' is stored as "X out of Y" and split into 'current_floor' and 'total_floors'.
- 'Bathroom', 'Balcony', 'Car Parking' are stored as strings and parsed to numeric.
- 'Plot Area' is a duplicate of 'Dimensions' and was dropped.
- ~63% of the raw rows (112,592 of 177,847) are exact duplicates from scraping and are dropped.
- Dataset contains free-text columns: 'Title', 'Description', 'Society', 'location'.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Amount(in rupees)",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "house_prices.csv")
print("Loaded data shape:", df.shape)

Loaded data shape: (187531, 21)


In [4]:
# Use if needed to get see all cols of pandas dataframes
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
df.head()

,Index,Title,Description,Amount(in rupees),Price (in rupees),location,Carpet Area,Status,Floor,Transaction,Furnishing,facing,overlooking,Society,Bathroom,Balcony,Car Parking,Ownership,Super Area,Dimensions,Plot Area
0,0,1 BHK Ready to Occupy Flat for sale in Srushti Siddhi Mangal Murti Complex Bhiwandi,"Bhiwandi, Thane has an attractive 1 BHK Flat for sale. The property is ideally located in a strategic location in Srushti Siddhi Mangal Murti Complex township. This flat for resale is a choice property. This apartment ready to move in the Bhiwandi is available for an attractive price of INR 42 Lac. You will find it unfurnished.",42 Lac,6000.0,thane,500 sqft,Ready to Move,10 out of 11,Resale,Unfurnished,NaN,NaN,Srushti Siddhi Mangal Murti Complex,1,2,NaN,NaN,NaN,NaN,NaN
1,1,2 BHK Ready to Occupy Flat for sale in Dosti Vihar Pokhran Road,"One can find this stunning 2 BHK flat for sale in Pokhran Road, Thane. It enjoys an excellent location within the Dosti Vihar. This flat for resale is a choice property. This ready to move flat in Pokhran Road can be availed at a reasonable price of INR 98 Lac. This semi-furnished flat is strategically designed with all the amenities to enhance the living experience. The property is strategically placed near prominent places as near singhaniya school which make for the smooth living of residents.",98 Lac,13799.0,thane,473 sqft,Ready to Move,3 out of 22,Resale,Semi-Furnished,East,Garden/Park,Dosti Vihar,2,NaN,1 Open,Freehold,NaN,NaN,NaN
2,2,2 BHK Ready to Occupy Flat for sale in Sunrise by Kalpataru Kolshet Road,"Up for immediate sale is a 2 BHK apartment in Kolshet Road, Thane. Don't miss this bargain flat for sale. Situated in the Sunrise By Kalpataru township, it has a prime location. This flat for resale has a desirable location. You can buy this ready to move flat in Kolshet Road at a reasonable price of INR 1.40 Cr. This unfurnished flat is strategically designed with all the amenities to enhance the living experience. Landmarks near the apartment include pokhran road no 2.",1.40 Cr,17500.0,thane,779 sqft,Ready to Move,10 out of 29,Resale,Unfurnished,East,Garden/Park,Sunrise by Kalpataru,2,NaN,1 Covered,Freehold,NaN,NaN,NaN
3,3,1 BHK Ready to Occupy Flat for sale Kasheli,"This beautiful 1 BHK Flat is available for sale in Kasheli, Thane. This flat for resale has a desirable location. This ready to move flat is offered at an economical price of INR 25 Lac. You will find it unfurnished.",25 Lac,NaN,thane,530 sqft,Ready to Move,1 out of 3,Resale,Unfurnished,NaN,NaN,NaN,1,1,NaN,NaN,NaN,NaN,NaN
4,4,2 BHK Ready to Occupy Flat for sale in TenX Habitat Raymond Realty Pokhran Road,"This lovely 2 BHK Flat in Pokhran Road, Thane is up for sale. This flat is situated in the Tenx Habitat Raymond Realty township and is equipped with premium facilities. This flat is an attractive property for resale. You can buy this ready to move flat in Pokhran Road at a reasonable price of INR 1.60 Cr. You will find it unfurnished. Some of the landmarks in the vicinity include pokhran road 2.",1.60 Cr,18824.0,thane,635 sqft,Ready to Move,20 out of 42,Resale,Unfurnished,West,"Garden/Park, Main Road",TenX Habitat Raymond Realty,2,NaN,1 Covered,Co-operative Society,NaN,NaN,NaN


In [6]:
df[["Plot Area", "Dimensions"]].drop_duplicates()


,Plot Area,Dimensions
0,NaN,NaN


In [7]:


# --- Drop row index,  "Plot Area", "Dimensions" as they are all null columns
df = df.drop(columns=["Index", "Plot Area", "Dimensions"])

# --- Target: "42 Lac " / "1.40 Cr " → numeric rupees ---
# 1 Lac = 100_000 ; 1 Cr = 10_000_000
def parse_amount(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip()
    if "Cr" in val:
        return float(val.replace("Cr", "").strip()) * 1e7
    elif "Lac" in val:
        return float(val.replace("Lac", "").strip()) * 1e5
    return np.nan

df["Amount(in rupees)"] = df["Amount(in rupees)"].apply(parse_amount)
df = df.dropna(subset=["Amount(in rupees)"]).reset_index(drop=True)

# --- Area columns: "500 sqft" → 500.0 ---
for col in ["Carpet Area", "Super Area"]:
    df[col] = pd.to_numeric(
        df[col].str.replace("sqft", "", regex=False).str.strip(), errors="coerce"
    )

# --- Floor: "10 out of 11" → current_floor=10, total_floors=11 ---
floor_parsed = df["Floor"].str.extract(r"(\d+|Ground)\s+out of\s+(\d+)", expand=True)
df["current_floor"] = pd.to_numeric(
    floor_parsed[0].str.replace("Ground", "0", regex=False), errors="coerce"
)
df["total_floors"] = pd.to_numeric(floor_parsed[1], errors="coerce")
df = df.drop(columns=["Floor"])

# --- Bathroom, Balcony: "2" / "> 10" → extract leading number ---
df["Bathroom"] = pd.to_numeric(df["Bathroom"], errors="coerce")
df["Balcony"] = pd.to_numeric(
    df["Balcony"].str.extract(r"(\d+)", expand=False), errors="coerce"
)

# --- Car Parking: "1 Covered" → 1 ---
df["Car Parking"] = pd.to_numeric(
    df["Car Parking"].str.extract(r"(\d+)", expand=False), errors="coerce"
)

# --- Free-text string columns: object → explicit StringDtype ---
for col in ["Title", "Description", "Society", "location"]:
    df[col] = df[col].astype("string")

# --- Low-cardinality categoricals → category dtype ---
for col in ["Status", "Transaction", "Furnishing", "facing", "overlooking", "Ownership"]:
    df[col] = df[col].astype("category")

# --- Deduplicate: ~63% of raw rows are exact scraping duplicates ---
n_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Dropped {n_before - len(df):,} duplicate rows ({(n_before - len(df)) / n_before:.1%})")

# --- Randomize sample order (curation guideline) ---
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Preprocessed shape:", df.shape)
print(df.dtypes)

Dropped 112,592 duplicate rows (63.3%)
Preprocessed shape: (65255, 19)
Title                string[python]
Description          string[python]
Amount(in rupees)           float64
Price (in rupees)           float64
location             string[python]
Carpet Area                 float64
Status                     category
Transaction                category
Furnishing                 category
facing                     category
overlooking                category
Society              string[python]
Bathroom                    float64
Balcony                     float64
Car Parking                 float64
Ownership                  category
Super Area                  float64
current_floor               float64
total_floors                float64
dtype: object


## Data Checks

In [8]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    target_feature=task_mold.target_column_name,
    problem_type=task_mold.problem_type,
    print_report=True, # In notebook...
)


#### Dataset Overview
Rows: 65,255
Columns: 19
Use sampling: False (sample size: 65,255)

#### Sample Rows
                                                                                    Title  \
0                                      2 BHK Ready to Occupy Flat for sale Yenamalakuduru   
1                             3 BHK Ready to Occupy Flat for sale in Opera Garden Dhakoli   
2                    3 BHK Ready to Occupy Flat for sale in Skylark Apartments Sector 115   
3  2 BHK Ready to Occupy Flat for sale in Royal Palms Diamond Isle Phase II Goregaon East   
4       3 BHK Ready to Occupy Flat for sale in Shalimar One World Vista Amar Shaheed Path   

                                                                                                                                                                                                                                                                                                                                             

In [9]:
# Sample Rows
df_head

,Title,Description,Amount(in rupees),Price (in rupees),location,Carpet Area,Status,Transaction,Furnishing,facing,overlooking,Society,Bathroom,Balcony,Car Parking,Ownership,Super Area,current_floor,total_floors
0,2 BHK Ready to Occupy Flat for sale Yenamalakuduru,This ready to move-in 2 BHK flat is available for sale at the premium Yenamalakuduru in Vijayawada. This flat for sale is a choice property. This ready to move flat in Yenamalakuduru comes at an affordable price of INR 27 Lac. This immaculate flat boasts of coming in furnished form which takes the entire deal to the next level.,2700000.0,3079.0,vijayawada,NaN,Ready to Move,New Property,Furnished,NaN,NaN,<NA>,2.0,1.0,NaN,NaN,877.0,2.0,5.0
1,3 BHK Ready to Occupy Flat for sale in Opera Garden Dhakoli,"Have a look at this immaculate 3 BHK flat for sale in Dhakoli, Zirakpur. Ideally situated in the Opera Garden township it enjoys a prime location. This is one of the best properties available for sale. This ready to move property in Dhakoli is readily available within an affordable cost of INR 99 Lac. The flat is semi-furnished and makes for an ideal choice for any family. The property is strategically placed near prominent places as panchkula shopping complex which make for the smooth living of residents.",9900000.0,5077.0,zirakpur,1450.0,Ready to Move,New Property,Semi-Furnished,North - East,"Pool, Garden/Park, Main Road",Opera Garden,3.0,3.0,NaN,Freehold,NaN,3.0,11.0
2,3 BHK Ready to Occupy Flat for sale in Skylark Apartments Sector 115,"This attractive 3 BHK apartment can be found for sale in Sector 115, Mohali. It is based at Skylark Apartments complex, that occupies a prominent place in the locality. This is one of the best properties available for resale. The ready to move flat in the prime area of Sector 115 is available at a reasonable price of INR 62 Lac.",6200000.0,3261.0,mohali,NaN,Ready to Move,Resale,NaN,NaN,NaN,Skylark Apartments,NaN,NaN,NaN,NaN,1901.0,NaN,NaN
3,2 BHK Ready to Occupy Flat for sale in Royal Palms Diamond Isle Phase II Goregaon East,2 BHK flat available for sale in Mumbai in the prime location of Goregaon East. It is housed in the well-planned Royal Palms Diamond Isle Phase Ii township in an advantageous location. This apartment is a property of choice for resale. This ready to move flat in Goregaon East can be taken at a very economical pricing of INR 65 Lac. This immaculate flat boasts of coming in semi-furnished form which takes the entire deal to the next level. The flat is in close proximity to prominent landmarks like film city and powai.,6500000.0,8333.0,mumbai,625.0,Ready to Move,Resale,Semi-Furnished,East,"Pool, Garden/Park, Main Road",Royal Palms Diamond Isle Phase II,2.0,NaN,NaN,NaN,NaN,8.0,16.0
4,3 BHK Ready to Occupy Flat for sale in Shalimar One World Vista Amar Shaheed Path,"Carefully laid out in the prime location of Amar Shaheed Path in Lucknow, this spacious 3 BHK flat on sale is a meticulously planned project. Strategically situated in the Shalimar One World Vista site, it is placed at a prime location. This is one of the best properties available for resale. This ready to move flat located in Amar Shaheed Path is available for purchase at a fair price of INR 1.35 Cr. The flat is available in unfurnished condition. The prime landmarks near this property are near by saheed path..",13500000.0,7377.0,lucknow,1500.0,Ready to Move,Resale,Unfurnished,North - West,"Garden/Park, Pool, Main Road",Shalimar One World Vista,3.0,3.0,1.0,Freehold,NaN,15.0,18.0


In [10]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,overlooking,category,27731.0,42.50,18.0,"Main Road, Garden/Park, Main Road, Garden/Park, Garden/Park, Pool, Main Road, Pool, Garden/Park, Main Road, Garden/Park, Pool, Pool, Garden/Park, Pool, Main Road, Garden/Park, Main Road, Garden/Park, Pool"
1,facing,category,25142.0,38.53,8.0,"East, North - East, North, West, South, South - East, North - West, South -West"
2,Ownership,category,23380.0,35.83,4.0,"Freehold, Leasehold, Co-operative Society, Power Of Attorney"
3,Furnishing,category,1166.0,1.79,3.0,"Semi-Furnished, Unfurnished, Furnished"
4,Status,category,279.0,0.43,1.0,Ready to Move
5,Transaction,category,63.0,0.10,4.0,"Resale, New Property, Other, Rent/Lease"
6,Super Area,float64,38142.0,58.45,2540.0,"1000.0, 1100.0, 1200.0, 900.0, 1250.0, 1500.0, 800.0, 1150.0, 1050.0, 1800.0"
7,Car Parking,float64,37354.0,57.24,150.0,"1.0, 2.0, 3.0, 4.0, 5.0, 10.0, 6.0, 8.0, 15.0, 12.0"
8,Carpet Area,float64,30099.0,46.13,2326.0,"1000.0, 1100.0, 900.0, 1200.0, 800.0, 1500.0, 1400.0, 950.0, 1300.0, 750.0"
9,Balcony,float64,17933.0,27.48,10.0,"2.0, 1.0, 3.0, 4.0, 5.0, 6.0, 10.0, 8.0, 7.0, 9.0"


In [11]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
Amount(in rupees),65255.0,1.067909e+07,6.292519e+07,100000.0,1.400300e+10
Price (in rupees),62298.0,6.806530e+03,4.454801e+04,0.0,6.700000e+06
Carpet Area,35156.0,1.302607e+03,5.192416e+03,1.0,7.092220e+05
Bathroom,64825.0,2.416059e+00,8.752169e-01,1.0,1.000000e+01
Balcony,47322.0,2.091754e+00,1.015274e+00,1.0,1.000000e+01
Car Parking,27901.0,3.863159e+00,3.304791e+01,1.0,9.990000e+02
Super Area,27113.0,1.387320e+03,7.766600e+02,1.0,9.450000e+03
current_floor,62395.0,4.377017e+00,4.449762e+00,0.0,2.000000e+02
total_floors,62395.0,8.607244e+00,6.977741e+00,1.0,2.000000e+02


In [12]:
# Categorical Feature Statistics
cat_stats

value  \
column      rank                                                                        
Description 1                                                                    <NA>   
            2     Multistorey apartment is available for sale. It is a good locati...   
            3     N B GRIHA PRAVESH is a G+4 commercial cum residential project lo...   
            4     Multistorey apartment is available for sale. It is a good locati...   
            5     Allowing you to enjoy natures richness, the Unimark Springfield ...   
Furnishing  1                                                          Semi-Furnished   
            2                                                             Unfurnished   
            3                                                               Furnished   
            4                                                                    <NA>   
Ownership   1                                                                Freehold   
            2                                                                    <NA>   
            3                                                               Leasehold   
            4                                                    Co-operative Society   
            5                                                       Power Of Attorney   
Society     1                                                                    <NA>   
            2                                                              RPS Savana   
            3                                                           Sushma Grande   
            4                                                         Sushma Valencia   
            5                                                            CRC Sublimis   
Status      1                                                           Ready to Move   
            2                                                                    <NA>   
Title       1                           3 BHK Ready to Occupy Flat for sale Sector 85   
            2                      3 BHK Ready to Occupy Flat for sale Vaishali Nagar   
            3                           3 BHK Ready to Occupy Flat for sale Jagatpura   
            4              3 BHK Ready to Occupy Flat for sale Vasna Bhayli Main Road   
            5         3 BHK Ready to Occupy Flat for sale in Sushma Valencia Zirakpur   
Transaction 1                                                                  Resale   
            2                                                            New Property   
            3                                                                    <NA>   
            4                                                                   Other   
            5                                                              Rent/Lease   
facing      1                                                                    <NA>   
            2                                                                    East   
            3                                                            North - East   
            4                                                                   North   
            5                                                                    West   
location    1                                                               bangalore   
            2                                                               faridabad   
            3                                                                 gurgaon   
            4                                                                 chennai   
            5                                                           greater-noida   
overlooking 1                                                                    <NA>   
            2                                                               Main Road   
            3                                                  Garden/Park,

In [13]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.0,181.341,0.599,3.959580e+15,0.646,log,2242665.3,2213127.4,lognormal


## Task Curation

In [14]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
    group_labels=task_mold.group_labels,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended splits: n_repeats=3, n_splits=3, test_size=None


In [15]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

splits = curation_recommendations.get_recommended_iid_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=splits,
)

## Export

In [16]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to magicbricks_house_price/019d8c46-c910-7ed7-8386-06cf1464547c
019d8c46-c910-7ed7-8386-06cf1464547c
2152039507602dd1b82faa9edd79d0603d85b2f41e6a61325160d62fca36071a
